# Semantic LLM Router — Load Balancing Simulation

Runs the real router logic against **simulated backends** with dynamic KV-cache load.
**No GPU needed.** Pure Python asyncio — free Colab CPU is sufficient.

**Run each cell top-to-bottom, or click Runtime → Run all.**

## Step 1 — Setup (safe to re-run)

In [ ]:
import os

# Remove previous clone if it exists so re-runs work cleanly
if os.path.exists('/content/semantic-llm-router'):
    !rm -rf /content/semantic-llm-router

!git clone https://github.com/yfan000/semantic-llm-router.git /content/semantic-llm-router
os.chdir('/content/semantic-llm-router')
print('Working directory:', os.getcwd())
!pip install -q fastapi pydantic numpy
print('Setup complete.')

## Step 2 — Run all 5 scenarios (~3 min total on free CPU)

### Scenario 1 — Default
Equal-weight users, Poisson arrivals. Baseline load distribution across 3 backends.

In [ ]:
!python tests/simulation.py --scenario default --requests 200 --concurrency 10

### Scenario 2 — Spike
First 20% of requests arrive instantly. Watch the 8B load spike, price inflate, and traffic shift.

In [ ]:
!python tests/simulation.py --scenario spike --requests 300 --concurrency 20

### Scenario 3 — SLA pressure
All users set min_accuracy=0.80. 8B (floor 0.50) is excluded; 13B and 70B share traffic.

In [ ]:
!python tests/simulation.py --scenario sla --requests 200 --concurrency 10

### Scenario 4 — Eco mode
energy_weight=0.40. 8B at 26.7 tok/J dominates over 70B at 2.1 tok/J.

In [ ]:
!python tests/simulation.py --scenario eco --requests 200 --concurrency 10

### Scenario 5 — Mixed users
1/3 cost-mode, 1/3 eco-mode, 1/3 accuracy-mode. Each backend serves its natural user group.

In [ ]:
!python tests/simulation.py --scenario mixed_users --requests 400 --concurrency 15

## Step 3 — Latency penalty demo
model-a bids 200ms but always delivers 800ms (4× overrun).
After 40 samples its penalty multiplier inflates to ~2.7× and it loses future auctions.

In [ ]:
import sys, os, tempfile
sys.path.insert(0, '/content/semantic-llm-router')

from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tracker = ReputationTracker(path=tempfile.mktemp(suffix='.json'))

def bid(mid, cost, lat, acc=0.80):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=lat,
                       estimated_accuracy=acc, estimated_energy_j=10.0,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)

pref = UserPreference(cost_weight=0.6, latency_weight=0.2, accuracy_weight=0.1, energy_weight=0.1)

print('BEFORE penalty:')
w = select([bid('model-a', 0.001, 200), bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {w.model_id}  (model-a looks cheapest)')

for _ in range(40):
    tracker.record_latency('model-a', bid_latency_ms=200, actual_latency_ms=800)

p = tracker.get_penalty_multiplier('model-a')
print(f'\nAFTER 40 slow responses:')
print(f'  Reliability:      {tracker.get_latency_reliability("model-a"):.3f}')
print(f'  Penalty:          {p:.2f}x')
print(f'  Effective cost:   ${0.001*p:.4f}  (was $0.0010)')
w = select([bid('model-a', 0.001, 200), bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {w.model_id}  (model-b wins despite higher base cost)')

## Step 4 — Accuracy penalty demo
model-a bids accuracy=0.92 but judge scores 0.45 every time.
After 40 samples: effective accuracy = 0.92 × 0.49 ≈ 0.45 → honest model wins.

In [ ]:
import tempfile
from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tracker = ReputationTracker(path=tempfile.mktemp(suffix='.json'))

def bid(mid, cost, acc):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=400,
                       estimated_accuracy=acc, estimated_energy_j=10.0,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)

pref = UserPreference(accuracy_weight=0.60, cost_weight=0.15, latency_weight=0.15, energy_weight=0.10)

print('BEFORE accuracy penalty:')
w = select([bid('model-a', 0.001, 0.92), bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {w.model_id}  (model-a looks most accurate)')

for _ in range(40):
    tracker.record_accuracy_bid('model-a', 'math', 'hard', bid_accuracy=0.92, judge_score=0.45)

d = tracker.get_accuracy_discount('model-a', 'math', 'hard')
print(f'\nAFTER 40 overbidding events:')
print(f'  Discount:           {d:.3f}')
print(f'  Effective accuracy: {0.92*d:.3f}  (was 0.92)')
w = select([bid('model-a', 0.001, 0.92), bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {w.model_id}  (model-b wins: 0.75 > {0.92*d:.2f} effective)')